# FISH spectral workflow tutorial: Sun17_A_01 / M01

This notebook uses **real observed spectra**, not synthetic data. The selection is 32 consecutive integrations from N_0001 (zero-based original rows 1040:1072), both AA/BB, and the full 65,536-channel native band. Read each step and inspect the arrays before proceeding.

Install FISH first and open this notebook with `examples/minimal_calibration` as the working directory. Original observations remain unchanged; the bundled data are compact, lossless slices. The first cells explain calibration. Later cells run the complete production sequence through baseline subtraction and the spectral table; only spatial gridding is omitted.

In [ ]:
from pathlib import Path
import json
import numpy as np
from scipy.interpolate import interp1d
from fastfish.calibration.extracted import rebuild_reference
from fastfish.processing.preview import calibrate
from calibrate import load_inputs, REST_GHZ

raw, f, times, provenance = load_inputs()
data = Path.cwd() / 'data'
print('SCI (time, channel, polarization):', raw.shape)
print('Midpoint MJD:', times[[0,-1]])
print('Exposure total [s]:', provenance['science_duration_s'])
print('Frequency range [MHz]:', f[[0,-1]]*1000)


## 1. Construct the temperature reference

SCI midpoints are UTOBS + EXPOSURE/2; `load_inputs` has already made this conversion. Do not repeat it. CAL times already denote midpoints.

$$D=\frac{P_{\rm ON}-P_{\rm OFF}}{20480}\times\frac{20480+28672}{2},\qquad T_{\rm psr}=T_{\rm cal}P_{\rm OFF}/\operatorname{smooth}_{n_{t,cal}}(D).$$

The original numbers 20480 and 28672 are retained from the calibration code. This expression is equivalent to multiplying ON−OFF by 1.2; it follows this pipeline's extracted-CAL convention. Tcal is smoothed over 16 frequency channels and interpolated onto CAL channels; float32 reference rounding is retained. `n_t_cal` filters ON−OFF along time. `n_t_apply` then independently filters the resulting Tpsr along time. The compact CAL/Tcal retains sufficient context for both default windows. For this extract use windows from 1 to 10.

### ON/OFF timing and the normalization

The official FAST document [FAST Digital Backends and Science Data, Section 4, p. 4](https://fast.bao.ac.cn/static/uploadfiles/%E5%B8%AE%E5%8A%A9%E6%96%87%E6%A1%A3/FAST_Digital_Backends_and_Science_Data.pdf) defines the ON/OFF setting unit as 4 ns. The original calibration code uses ON=20480 and OFF=28672, corresponding to 81.92 μs and 114.688 μs. Their sum is 196.608 μs; the Sun17 CAL header gives TBIN=98.304 μs, half that cycle. The CRAFTS [pipeline paper, Section II.1](https://arxiv.org/html/2412.08173v2#S2.SS1) also documents the unequal noise-injection durations.

The extractor averages alternating equal-duration PSR samples separately. With the assumed alignment, the ON-labelled bin contains 81.92 μs of injection within 98.304 μs; its excess is diluted to 5/6. Thus the correction is t_sample/t_ON = 98.304/81.92 = (20480+28672)/(2×20480) = 1.2. It is not simply the inverse of the whole-cycle ON duty fraction. This interpretation assumes a diode pulse confined to the ON-labelled bin; the two timing values alone do not establish phase alignment. Check the observation setup for other datasets.


In [ ]:
n_t_cal = 10
n_t_apply = 10
assert 1 <= n_t_cal <= 10 and 1 <= n_t_apply <= 10
pack = rebuild_reference([data/'on_off.fits'], float(times[0]), float(times[-1]), f,
    data/'tcal_reference.fits', 'M01', n_t=n_t_apply, n_t_cal=n_t_cal)
print('Reference (time, polarization, frequency):', pack['raw'].shape)
print('Raw band-median Tpsr [K]:', np.median(pack['raw'],axis=(0,2)))


## 2. Match the science timestamps

Interpolate the filtered temperature reference to SCI midpoint times. This does not time-average the science spectra. `bounds_error=True` prevents time extrapolation.

In [ ]:
tpsr = interp1d(pack['mjd'], pack['smoothed'], axis=0, bounds_error=True)(times)
print('Reference on SCI times:', tpsr.shape)
print('Nonfinite reference fraction AA/BB:', np.mean(~np.isfinite(tpsr),axis=(0,2)))
print('Nonpositive reference fraction AA/BB:', np.mean(tpsr<=0,axis=(0,2)))


Real-data caveat: a few nonpositive reference values occur at 1420.28808594 MHz, inside the excluded H I gain interval. The production gain fit does not use that channel as an anchor. Inspect the reported fraction; do not call these values RFI solely because they are nonpositive.

## 3. Detect candidate RFI and estimate gain

Call the same `calibrate` function used by production previews and batch processing. Its chunk RFI search treats catalogue entries as priors. The gain estimator overlap-averages SCI onto CAL bins and fits cubic log(counts/Tpsr) independently for each SCI time and polarization. Gain anchors exclude H I within the production ±700 km/s interval and detected RFI.

There is no pointing/Doppler calculation in this calibration-only example: `scales=1` is explicit and all plotted velocities remain **topocentric**, not LSRK. The packaged search may return no intervals for this chunk; that is not proof of clean data.

In [ ]:
cfg = dict(rest_freq_ghz=REST_GHZ, gain_degree=3, protect_kms=250., rfi_catalog='packaged')
result = calibrate(raw, f, tpsr, pack['frequency_ghz'], np.ones(len(times)), [], cfg)
temperature = result['pol']
print('Calibrated (time, channel, polarization):', temperature.shape)
print('Gain coefficients:', result['coefficients'].shape)
print('Detected RFI intervals [MHz]:', result['intervals'])
print('Median calibrated temperature AA/BB [K]:', np.median(temperature,axis=(0,1)))


## 4. Verify against the original-input calculation

The compact input was checked against calibration using three surrounding original CAL files and the full original Tcal. At the default smoothing settings both effective temperature references and all calibrated SCI samples were exactly equal (maximum difference 0 K).

There is no known true astronomical spectrum. The frozen values below are a **repeatability reference**, not a calibration-accuracy standard. Non-default filter choices should be inspected as scientific changes and are not required to reproduce the default answer.

In [ ]:
expected = json.loads((Path.cwd()/'expected/regression_reference.json').read_text())
indices = expected['channel_indices']
assert np.isfinite(temperature).all()
if n_t_cal == 10 and n_t_apply == 10:
    np.testing.assert_allclose(temperature[:,indices,:], expected['temperature_K'], atol=1e-6, rtol=1e-8)
    np.testing.assert_allclose(result['coefficients'], expected['coefficients'], atol=1e-8, rtol=1e-8)
    print('Default regression checks passed')
else:
    difference = temperature[:,indices,:] - np.asarray(expected['temperature_K'])
    print('Maximum sampled change from default [K]:', np.max(abs(difference)))


## 5. Explicitly save the result and diagnostic figure

The readable script repeats the steps above, verifies input hashes, and saves outputs to a separate directory. The first two panels show raw (faint) and channel-wise time-filtered (solid) reference summaries. Spectral panels use display-only 32-channel means; calibration retains all native channels. No baseline or continuum is removed.

In [ ]:
import tempfile
from calibrate import run
from IPython.display import Image, display
output = Path(tempfile.mkdtemp(prefix='fish_suna01_calibration_'))
summary = run(output, n_t_cal=n_t_cal, n_t_apply=n_t_apply)
display(Image(filename=str(output/'calibration.png')))
print('Output folder:', output)


## Parameter tests and code inspection

- Set `n_t_cal=1` or `n_t_apply=1` and rerun the cells. Compare the reference curves, gain coefficients and resulting spectra with the default calculation. The two filters act on different quantities.
- Read `rebuild_reference` in `src/fastfish/calibration/extracted.py` and locate the duty factor, filters, diode interpolation and float32 rounding.
- Read `calibrate` in `src/fastfish/processing/preview.py` and `frequency_gain` in `src/fastfish/processing/spectral.py`. Identify RFI tracking, H I exclusion, overlap averaging, log-polynomial gain and support checks.
- Larger smoothing windows need a longer CAL extract; do not extrapolate or remove the context check just to run a larger value.

The following cells extend this calibration explanation through baseline removal and resampling. Absolute calibration accuracy and the true baseline beneath protected H I require independent scientific assessment.

## 6. Baseline specification and observed RFI

The production calibration stage saves observed RFI intervals per chunk. Catalogue entries alone are not masks. Those intervals, protected H I, and any extra exclusions determine the baseline anchors. This sample has no current catalogue-guided detections; that is not proof of no interference.

The initial baseline is an ordered full-band sin/cos component followed by a local linear residual within ±1500 km/s, protecting ±250 km/s. AA/BB are fitted separately. The local component is neither fitted nor subtracted outside its specified range. Models are evaluated inside protected H I but H I never anchors them. Change one specification at a time and inspect the effect.

In [ ]:
components = [
    dict(function='sincos', range_kms=None, period_MHz=None),
    dict(function='polynomial', degree=1, range_kms=[-1500.,1500.])
]
print(components)


## 7. Execute the full four-stage workflow

This calls the normal FISH run/wait commands: temperature reference → calibration/RFI → baseline → Doppler-aligned resampling/table. It uses the same real spectra and saved actual LSRK factors, in two 16-row chunks and one worker. Generated files are saved outside the example. No cube is made.

The defaults are an example configuration, not a uniquely valid baseline solution. A reported period-search boundary must be inspected. Only the default calculation is shown in the included expected figures.

In [ ]:
from run_workflow import run as run_spectral_workflow
workflow_output = Path(tempfile.mkdtemp(prefix='fish_suna01_workflow_'))
workflow_summary = run_spectral_workflow(workflow_output,
    n_t_cal=n_t_cal, n_t_apply=n_t_apply, baseline_components=components)
print('Workflow output:', workflow_output)


## 8. Inspect saved baseline fits and residuals

These figures reconstruct the **saved** per-polarization fits; no independent fit was made just for presentation. Native residual scatter contains radiometer noise. The spectral display averages 32 channels and clips the H I peak to expose smaller baseline residuals. The baseline beneath the protected line is model-dependent; a low residual outside H I does not establish its true error inside H I.

In [ ]:
display(Image(filename=str(workflow_output/'baseline.png')))
print('Chunks with period at search limit:', workflow_summary['baseline_period_at_limit_chunks'])
print('First native-spectrum residual statistics:', workflow_summary['baseline_first_spectrum'])


## 9. Load the final spectral table

The table has a spectrum row for each selected timestamp, on one common LSRK frequency grid. It is not a spatial map. Default n_b=2 and ±250 km/s give 32×2485 values. Changing baseline or resampling settings creates a different stage fingerprint; preserve comparison results.

In [ ]:
manifest = json.loads((workflow_output/'table_manifest.json').read_text())
processed = json.loads(Path(manifest['processed']).read_text())
product = processed[0]['products'][0]
table = np.load(product['data'], mmap_mode='r')
frequency_out = np.asarray(manifest['frequency_ghz'])
velocity_out = (1-frequency_out/REST_GHZ)*299792.458
print('Table:',table.shape,'all finite:',np.isfinite(table).all())
print('RFI fraction field:',product['rfi'])
display(Image(filename=str(workflow_output/'table-spectrum.png')))


## 10. Continue in the actual viewer

Install the Qt extra, then open the saved session in a terminal:

```bash
fish-viewer --session-state /your/workflow/output/fish_session.json
```

Inspect Tsys calibration, Apply calibration, Baseline and Table. After editing, Update preview applies settings; preview alone does not overwrite prior stage products. Restore the same session in FISH, then run/wait the necessary stages to save a new result. The compact example allows all four spectral stages and blocks only spatial gridding. A full mapping example requires broader spatial coverage.